In [1]:

import os
import cv2
import numpy as np
import shutil
import yaml
import random
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
 

In [3]:
# ─────────────────────────────────────────────
# 🔧 CONFIGURATION — Edit these before running
# ─────────────────────────────────────────────
INPUT_DIR = r"F:\Final Indian Footpath Damage Segmentation Dataset1"  # ← parent folder; all subfolders (Train/Test/Valid) scanned automatically
OUTPUT_DIR = r"F:\yolo_footpath_dataset"
 
# Mask settings
MASK_BINARY_THRESHOLD = 127       # Pixels above this → foreground (damage)
MIN_CONTOUR_AREA = 800            # Ignore tiny noise blobs (pixels²)
MAX_POLYGON_POINTS = 50           # Downsample dense polygons to this many points
EPSILON_APPROX_FACTOR = 0.001     # Douglas-Peucker simplification (lower = more detail)
 
# Dataset split
TRAIN_RATIO = 0.80
RANDOM_SEED = 42
 
# Resize (set to None to keep original 6000×4000)
# Recommended: resize to speed up training. E.g. (1280, 853) keeps 3:2 ratio
RESIZE_TO = (1280, 853)   # (width, height) or None
 
# Class definition
CLASS_NAMES = ["damage"]   # Index 0 = damage
 
# ─────────────────────────────────────────────
# STEP 1: Find all paired image/mask files
# ─────────────────────────────────────────────
def find_pairs(input_dir: str) -> list[tuple[Path, Path]]:
    """
    Recursively scans input_dir AND ALL subfolders for jpg/png pairs.
    Matches by stem name: image001.jpg ↔ image001.png
    Handles Train/, Test/, Valid/ subfolders automatically.
    """
    input_path = Path(input_dir)
 
    # rglob = recursive glob → finds files in ALL subfolders
    jpg_files = {f.stem: f for f in input_path.rglob("*.jpg")}
    jpg_files.update({f.stem: f for f in input_path.rglob("*.JPG")})
    png_files = {f.stem: f for f in input_path.rglob("*.png")}
    png_files.update({f.stem: f for f in input_path.rglob("*.PNG")})
 
    print(f"\n   📂 Scanning recursively: {input_path}")
    print(f"   🖼️  JPG files found : {len(jpg_files)}")
    print(f"   🎭 PNG files found : {len(png_files)}")
 
    pairs = []
    unmatched = []
    for stem, jpg_path in jpg_files.items():
        if stem in png_files:
            pairs.append((jpg_path, png_files[stem]))
        else:
            unmatched.append(jpg_path.name)
 
    if unmatched:
        print(f"\n   ⚠️  {len(unmatched)} JPG(s) had no matching PNG mask — skipped:")
        for name in unmatched[:10]:
            print(f"      - {name}")
        if len(unmatched) > 10:
            print(f"      ... and {len(unmatched) - 10} more")
 
    print(f"\n✅ Found {len(pairs)} valid image/mask pairs")
    return pairs
 
 
# ─────────────────────────────────────────────
# STEP 2: Clean mask
# ─────────────────────────────────────────────
def clean_mask(mask_bgr: np.ndarray) -> np.ndarray:
    """
    Converts a BGR mask image to a clean binary mask.
    - Handles grayscale and RGB masks
    - Removes small noise contours
    - Returns uint8 binary mask: 0 = background, 255 = damage
    """
    # Convert to grayscale if RGB
    if len(mask_bgr.shape) == 3:
        gray = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2GRAY)
    else:
        gray = mask_bgr.copy()
 
    # Binarize
    _, binary = cv2.threshold(gray, MASK_BINARY_THRESHOLD, 255, cv2.THRESH_BINARY)
 
    # Morphological cleanup: close small gaps, remove isolated pixels
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
 
    # Remove small blobs
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean = np.zeros_like(binary)
    for cnt in contours:
        if cv2.contourArea(cnt) >= MIN_CONTOUR_AREA:
            cv2.drawContours(clean, [cnt], -1, 255, thickness=cv2.FILLED)
 
    return clean
 
 
# ─────────────────────────────────────────────
# STEP 3: Mask → YOLO polygon labels
# ─────────────────────────────────────────────
def mask_to_yolo_labels(
    binary_mask: np.ndarray,
    img_width: int,
    img_height: int,
    class_id: int = 0
) -> list[str]:
    """
    Converts a binary mask to YOLO segmentation label lines.
    Each contour becomes one polygon.
 
    Returns list of strings like:
        "0 x1 y1 x2 y2 x3 y3 ..."
    where coordinates are normalized to [0, 1].
    """
    contours, hierarchy = cv2.findContours(
        binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
 
    label_lines = []
 
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_CONTOUR_AREA:
            continue
 
        # Douglas-Peucker polygon approximation (reduces point count)
        epsilon = EPSILON_APPROX_FACTOR * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
 
        # Flatten: shape (N, 1, 2) → (N, 2)
        points = approx.reshape(-1, 2)
 
        # Further downsample if still too many points
        if len(points) > MAX_POLYGON_POINTS:
            indices = np.linspace(0, len(points) - 1, MAX_POLYGON_POINTS, dtype=int)
            points = points[indices]
 
        # Need at least 3 points for a polygon
        if len(points) < 3:
            continue
 
        # Normalize coordinates to [0, 1]
        norm_points = []
        for x, y in points:
            nx = round(float(x) / img_width, 6)
            ny = round(float(y) / img_height, 6)
            # Clamp to [0, 1] for safety
            nx = max(0.0, min(1.0, nx))
            ny = max(0.0, min(1.0, ny))
            norm_points.extend([nx, ny])
 
        coord_str = " ".join(map(str, norm_points))
        label_lines.append(f"{class_id} {coord_str}")
 
    return label_lines
 
 
# ─────────────────────────────────────────────
# STEP 4: Process a single pair
# ─────────────────────────────────────────────
def process_pair(
    jpg_path: Path,
    png_path: Path,
    out_img_dir: Path,
    out_lbl_dir: Path,
    resize_to: tuple | None = None
) -> bool:
    """
    Reads one image/mask pair, generates the YOLO label file,
    and saves both image and label to the output directories.
 
    Returns True if successful, False if mask had no valid contours.
    """
    # Read image
    img = cv2.imread(str(jpg_path))
    if img is None:
        print(f"  ❌ Could not read image: {jpg_path.name}")
        return False
 
    # Read mask
    mask_raw = cv2.imread(str(png_path))
    if mask_raw is None:
        print(f"  ❌ Could not read mask: {png_path.name}")
        return False
 
    # Resize if configured
    if resize_to is not None:
        img = cv2.resize(img, resize_to, interpolation=cv2.INTER_LINEAR)
        mask_raw = cv2.resize(mask_raw, resize_to, interpolation=cv2.INTER_NEAREST)
 
    h, w = img.shape[:2]
 
    # Clean mask
    binary_mask = clean_mask(mask_raw)
 
    # Convert to YOLO polygons
    label_lines = mask_to_yolo_labels(binary_mask, w, h, class_id=0)
 
    # Write label file even if empty (so YOLO knows it's a "background" image)
    stem = jpg_path.stem
    label_path = out_lbl_dir / f"{stem}.txt"
    with open(label_path, "w") as f:
        f.write("\n".join(label_lines))
 
    # Save image as JPG
    out_img_path = out_img_dir / f"{stem}.jpg"
    cv2.imwrite(str(out_img_path), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
 
    return len(label_lines) > 0
 
 
# ─────────────────────────────────────────────
# STEP 5: Build output directory structure
# ─────────────────────────────────────────────
def create_output_dirs(output_dir: str) -> dict:
    base = Path(output_dir)
    dirs = {
        "train_img": base / "images" / "train",
        "val_img":   base / "images" / "val",
        "train_lbl": base / "labels" / "train",
        "val_lbl":   base / "labels" / "val",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs
 
 
# ─────────────────────────────────────────────
# STEP 6: Write dataset.yaml
# ─────────────────────────────────────────────
def write_yaml(output_dir: str, class_names: list[str]):
    yaml_data = {
        "path": output_dir,
        "train": "images/train",
        "val": "images/val",
        "nc": len(class_names),
        "names": {i: name for i, name in enumerate(class_names)},
    }
    yaml_path = Path(output_dir) / "dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)
    print(f"\n📄 dataset.yaml written → {yaml_path}")
    return yaml_path
 
 
# ─────────────────────────────────────────────
# STEP 7: Class balance report
# ─────────────────────────────────────────────
def report_balance(label_dirs: list[Path], split_names: list[str]):
    print("\n📊 Class Balance Report:")
    print(f"  {'Split':<10} {'Total':<10} {'w/ damage':<12} {'empty':<10}")
    print("  " + "-" * 42)
    for ldir, name in zip(label_dirs, split_names):
        txt_files = list(ldir.glob("*.txt"))
        total = len(txt_files)
        with_damage = sum(1 for f in txt_files if f.read_text().strip() != "")
        empty = total - with_damage
        pct = (with_damage / total * 100) if total else 0
        print(f"  {name:<10} {total:<10} {with_damage:<12} {empty:<10}  ({pct:.1f}% damage)")
 
 
# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  Footpath Damage → YOLO Segmentation Preprocessing")
    print("=" * 60)
 
    # Validate input
    if not Path(INPUT_DIR).exists():
        raise FileNotFoundError(f"Input directory not found:\n  {INPUT_DIR}\n"
                                f"Please check your pendrive is connected and path is correct.")
 
    # 1. Find pairs
    pairs = find_pairs(INPUT_DIR)
    if not pairs:
        raise ValueError("No valid image/mask pairs found. Check filenames match (same stem).")
 
    # 2. Shuffle and split
    random.seed(RANDOM_SEED)
    random.shuffle(pairs)
    split_idx = int(len(pairs) * TRAIN_RATIO)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]
    print(f"   Train: {len(train_pairs)} pairs")
    print(f"   Val:   {len(val_pairs)} pairs")
 
    # 3. Create output directories
    dirs = create_output_dirs(OUTPUT_DIR)
 
    # 4. Process train set
    print("\n🔄 Processing TRAIN set...")
    train_ok, train_empty = 0, 0
    for jpg, png in tqdm(train_pairs, unit="pair"):
        result = process_pair(jpg, png, dirs["train_img"], dirs["train_lbl"], RESIZE_TO)
        if result:
            train_ok += 1
        else:
            train_empty += 1
 
    # 5. Process val set
    print("\n🔄 Processing VAL set...")
    val_ok, val_empty = 0, 0
    for jpg, png in tqdm(val_pairs, unit="pair"):
        result = process_pair(jpg, png, dirs["val_img"], dirs["val_lbl"], RESIZE_TO)
        if result:
            val_ok += 1
        else:
            val_empty += 1
 
    # 6. Write YAML
    write_yaml(OUTPUT_DIR, CLASS_NAMES)
 
    # 7. Reports
    report_balance(
        [dirs["train_lbl"], dirs["val_lbl"]],
        ["train", "val"]
    )
 
    print("\n✅ Preprocessing complete!")
    print(f"   Train: {train_ok} with damage labels, {train_empty} background-only")
    print(f"   Val:   {val_ok} with damage labels, {val_empty} background-only")
    print(f"\n📁 Output saved to: {OUTPUT_DIR}")
    print("\n🚀 Next step — train YOLOv8 segmentation:")
    print("   pip install ultralytics")
    print(f"   yolo segment train data={OUTPUT_DIR}/dataset.yaml model=yolov8m-seg.pt epochs=100 imgsz=1280 batch=4")
 
 
if __name__ == "__main__":
    main()

  Footpath Damage → YOLO Segmentation Preprocessing

   📂 Scanning recursively: F:\Final Indian Footpath Damage Segmentation Dataset1
   🖼️  JPG files found : 583
   🎭 PNG files found : 583

✅ Found 583 valid image/mask pairs
   Train: 466 pairs
   Val:   117 pairs

🔄 Processing TRAIN set...


100%|██████████| 466/466 [08:41<00:00,  1.12s/pair]



🔄 Processing VAL set...


100%|██████████| 117/117 [02:14<00:00,  1.15s/pair]



📄 dataset.yaml written → F:\yolo_footpath_dataset\dataset.yaml

📊 Class Balance Report:
  Split      Total      w/ damage    empty     
  ------------------------------------------
  train      480        473          7           (98.5% damage)
  val        133        128          5           (96.2% damage)

✅ Preprocessing complete!
   Train: 459 with damage labels, 7 background-only
   Val:   112 with damage labels, 5 background-only

📁 Output saved to: F:\yolo_footpath_dataset

🚀 Next step — train YOLOv8 segmentation:
   pip install ultralytics
   yolo segment train data=F:\yolo_footpath_dataset/dataset.yaml model=yolov8m-seg.pt epochs=100 imgsz=1280 batch=4
